In [1]:
!pip install torch torchvision transformers
!pip install scikit-learn pandas numpy tqdm emoji

In [2]:
import os
import re
import emoji
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from collections import Counter

import torch
from transformers import CLIPProcessor, CLIPModel

In [3]:
df = pd.read_csv("/kaggle/input/notebooks/navishaaagarwaal/01-dataset-preparation/labeled.csv")
print("Dataset size:", len(df))
df.head()

Dataset size: 11902


,text,image_path,label
0,Knocked doors with the venerable #TeamTrudeau ...,/kaggle/input/datasets/vincemarcs/mvsamultiple...,0
1,An NPD gov't would institutionalize mediocrity...,/kaggle/input/datasets/vincemarcs/mvsamultiple...,1
2,"""""I think it's time for change"""" - Ana Commit ...",/kaggle/input/datasets/vincemarcs/mvsamultiple...,0
3,The Past and Future of the Refugee Crisis - Th...,/kaggle/input/datasets/vincemarcs/mvsamultiple...,0
4,Rdy to watch @ThomasMulcair rock it tnight in ...,/kaggle/input/datasets/vincemarcs/mvsamultiple...,0


In [4]:
def clean_text(text):

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = emoji.demojize(text)

    return text.lower().strip()


def extract_hashtags(text):

    return re.findall(r"#\w+", text)


def segment_hashtag(tag):

    tag = tag.replace("#", "")
    tag = re.sub(r'([a-z])([A-Z])', r'\1 \2', tag)
    tag = re.sub(r'(\d+)', r' \1 ', tag)

    return tag.lower()

In [5]:
text_inputs = []
hashtag_inputs = []
image_paths = []
all_hashtags = []

for _, row in df.iterrows():

    clean = clean_text(row["text"])

    hashtags = extract_hashtags(row["text"])
    segmented = [segment_hashtag(tag) for tag in hashtags]

    hashtag_text = " ".join(segmented) if segmented else "none"

    text_inputs.append(clean)
    hashtag_inputs.append(hashtag_text)
    image_paths.append(row["image_path"])
    all_hashtags.append(hashtags)

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

model.eval()

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [7]:
def get_text_embeddings(text_list, batch_size=32):

    embeddings = []

    for i in tqdm(range(0, len(text_list), batch_size)):

        batch = text_list[i:i+batch_size]

        inputs = processor(
            text=batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        with torch.no_grad():

            outputs = model.text_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            pooled = outputs.pooler_output
            features = model.text_projection(pooled)

        embeddings.append(features.cpu())

    return torch.cat(embeddings).numpy()

In [8]:
def get_image_embeddings(image_paths, batch_size=32):

    embeddings = []

    for i in tqdm(range(0, len(image_paths), batch_size)):

        batch_paths = image_paths[i:i+batch_size]
        images = []

        for path in batch_paths:

            try:
                img = Image.open(path).convert("RGB")
            except:
                img = Image.new("RGB",(224,224))

            images.append(img)

        inputs = processor(images=images, return_tensors="pt")

        pixel_values = inputs["pixel_values"].to(device)

        with torch.no_grad():

            outputs = model.vision_model(pixel_values=pixel_values)
            pooled = outputs.pooler_output
            features = model.visual_projection(pooled)

        embeddings.append(features.cpu())

    return torch.cat(embeddings).numpy()

In [9]:
print("Encoding text...")
text_emb = get_text_embeddings(text_inputs)

print("Encoding hashtags...")
hash_emb = get_text_embeddings(hashtag_inputs)

print("Encoding images...")
img_emb = get_image_embeddings(image_paths)

Encoding text...


100%|██████████| 372/372 [00:12<00:00, 30.36it/s]


Encoding hashtags...


100%|██████████| 372/372 [00:05<00:00, 63.27it/s]


Encoding images...


100%|██████████| 372/372 [00:35<00:00, 10.46it/s]


In [10]:
flat_tags = [tag for tags in all_hashtags for tag in tags]

freq_dict = Counter(flat_tags)

hashtag_count = np.array([len(tags) for tags in all_hashtags])

hashtag_freq = np.array([
    sum(freq_dict[t] for t in tags) if tags else 0
    for tags in all_hashtags
])

In [11]:
np.save("text_emb.npy", text_emb)
np.save("hash_emb.npy", hash_emb)
np.save("img_emb.npy", img_emb)

np.save("hashtag_count.npy", hashtag_count)
np.save("hashtag_freq.npy", hashtag_freq)

print("Feature extraction complete.")

Feature extraction complete.


In [12]:
print(text_emb.shape)
print(img_emb.shape)
print(hash_emb.shape)
print(hashtag_count.shape)
print(hashtag_freq.shape)

(11902, 512)
(11902, 512)
(11902, 512)
(11902,)
(11902,)
